In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup


In [2]:
# o projeto deve listar os sites relevantes a serem buscados assim como o projeto original tem uma lista de opçoes.

In [3]:

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4o-mini"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [4]:
system_message = """Você é um assistente útil que pode buscar e resumir notícias da internet.
Quando o usuário pedir para buscar notícias de um site específico, use a função create_brochure.
Você também pode conversar normalmente sobre outros assuntos."""

In [5]:
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:
    """
    Uma classe utilitária para representar um site que foi raspado (scrapeado), agora incluindo os links.
    """

    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers=headers)
        self.body = response.content
        soup = BeautifulSoup(self.body, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        if soup.body:
            for irrelevant in soup.body(["script", "style", "img", "input"]):
                irrelevant.decompose()
            self.text = soup.body.get_text(separator="\n", strip=True)
        else:
            self.text = ""
        links = [link.get('href') for link in soup.find_all('a')]
        self.links = [link for link in links if link]

    def get_contents(self):
        return f"Webpage Title:\n{self.title}\nWebpage Contents:\n{self.text}\n\n"

In [6]:
link_system_prompt = "Você recebe uma lista de links encontrados em uma página da web.\
Você deve decidir quais desses links seriam mais relevantes para incluir em um \
resumo das notícias que encontrara nos links. Você deve ignorar links que não são relevantes como de serviço e produtos e só trazer o resumo da noticia."
link_system_prompt += "Você deve responder em JSON conforme o exemplo:"
link_system_prompt += """
{
  "links": [
    {"type": "news article", "url": "https://exemplo.com/noticia1"}
  ]
}
Limite a 5 links mais relevantes.
"""

In [7]:
print(link_system_prompt)

Você recebe uma lista de links encontrados em uma página da web.Você deve decidir quais desses links seriam mais relevantes para incluir em um resumo das notícias que encontrara nos links. Você deve ignorar links que não são relevantes como de serviço e produtos e só trazer o resumo da noticia.Você deve responder em JSON conforme o exemplo:
{
  "links": [
    {"type": "news article", "url": "https://exemplo.com/noticia1"}
  ]
}
Limite a 5 links mais relevantes.



In [8]:
def get_links_user_prompt(website):
    user_prompt = (
        f"Aqui está uma lista de links do website {website.url}.\n"
        "Por favor, decida quais desses links são relevantes para um resumo de notícias e responda apenas com o URL completo (https) no formato JSON.\n"
        "Inclua somente links relacionados a notícias, artigos  — exclua todos os outros que tem a ver com produtos e serviços.\n\n"
        "Links:\n"
    )
    user_prompt += "\n".join(website.links[:50])
    return user_prompt


In [9]:
def get_links(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(website)}
      ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    return json.loads(result)

In [10]:
def get_all_details(url, max_links=40, max_chars_per_page=2000):
    result = "Landing page:\n"
    result += Website(url).get_contents()[:max_chars_per_page]

    links = get_links(url)
    print("Found links:", links)

    for link_url in links["links"][:max_links]:
        try:
            page = Website(link_url)
            result += f"\n\n{link_url}\n"  # mostra a URL
            result += page.get_contents()[:max_chars_per_page]  # pega o conteúdo
        except Exception as e:
            result += f"[Erro ao acessar {link_url}: {e}]\n"

    return result


In [11]:
system_prompt = "Você é um assistente que analisa o conteúdo de várias páginas relevantes do site de notícias \
e cria um breve folheto sobre as principais notícias do dia.  \
Inclua detalhes sobre as notícias."

In [46]:
def create_brochure(url):
    """Cria um resumo de notícias a partir de uma URL"""
    try:
        user_prompt = (
            f"Analise o seguinte conteúdo da web e crie um resumo das principais notícias "
            f"encontradas. Organize em tópicos claros e seja conciso.\n\n"
        )
        user_prompt += get_all_details(url)
        
        response = openai.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "Você é um especialista em resumir notícias de forma clara e objetiva."},
                {"role": "user", "content": user_prompt}
            ],
        )
        print(response.choices[0].message.content)
        return response.choices[0].message.content
    except Exception as e:
        return f"Erro ao criar resumo: {str(e)}"


In [48]:

pythonSITE_ADDRESSES = {
    # Sites que você já tem
    "uol": "https://noticias.uol.com.br/",
    "bbc": "https://www.bbc.com/portuguese",
    "dw": "https://www.dw.com/pt-br/notícias/s-7111",
    
    # Sites brasileiros que geralmente funcionam
    "g1": "https://g1.globo.com/",
    "r7": "https://noticias.r7.com/",
    "estadao": "https://www.estadao.com.br/",
    "folha": "https://www1.folha.uol.com.br/",
    "oglobo": "https://oglobo.globo.com/",
    "valor": "https://valor.globo.com/",
    "exame": "https://exame.com/",
    "infomoney": "https://www.infomoney.com.br/",
    "cnnbrasil": "https://www.cnnbrasil.com.br/",
    "band": "https://www.band.uol.com.br/noticias",
    "sbt": "https://www.sbtnews.com.br/",
    
    # Sites internacionais em português que funcionam bem
    "euronews": "https://pt.euronews.com/",
    "rfi": "https://www.rfi.fr/br/",
    "voanews": "https://www.voanews.com/pt",
    
    # Sites especializados
    "tecnoblog": "https://tecnoblog.net/",
    "olhardigital": "https://olhardigital.com.br/",
    "canaltech": "https://canaltech.com.br/",
    
    # Sites de economia
    "moneytimes": "https://moneytimes.com.br/",
    "seu_dinheiro": "https://www.seudinheiro.com/",
    
    # Sites de esportes
    "ge": "https://ge.globo.com/",
    "espn": "https://www.espn.com.br/",
    "lance": "https://www.lance.com.br/",
    
    # Sites alternativos/independentes
    "cartacapital": "https://www.cartacapital.com.br/",
    "theintercept": "https://www.intercept.com.br/",
    "nexo": "https://www.nexojornal.com.br/"
}

def adress_function(destination_adress):
    site = destination_adress.lower()
    return SITE_ADDRESSES.get(site, "Unknown")
    

Buscando endereço do site: g1
ChatCompletion(id='chatcmpl-CRM0zycLdoZ0lLcWZIMfnUTIhWmu2', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_zYILix1V9ZaQ2HL8Rm5pRIiM', function=Function(arguments='{"site_name":"g1.globo.com"}', name='adress_function'), type='function')]))], created=1760635517, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_560af6e559', usage=CompletionUsage(completion_tokens=20, prompt_tokens=646, total_tokens=666, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
Buscando endereço do site: g1.globo.com
ChatCompletion(id='chatcmpl-CRM10JN75Or6l4

In [22]:
news_function = {
    "name": "create_brochure",
    "description": (
        "Gera um resumo de notícias a partir de um site. "
        "Sempre use apenas após obter o endereço com a função get_adress, "
        "mesmo que o usuário forneça a URL diretamente."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "A URL do site onde buscar as notícias (deve começar com http:// ou https://)",
            },
        },
        "required": ["url"],
        "additionalProperties": False
    }
}

In [32]:
'''
O modelo lê isso internamente e entende:
Existe uma função chamada get_adress(destination_adress: string) que me dá o endereço do site.
Posso chamá-la se o usuário perguntar algo relacionado a notícias.
'''
site_function = {
    "name": "adress_function",
    "description": "Encontra o endereço do site de busca de notícias. Na sua resposta você deve apenas replicar o endereço do site.",
    "parameters": {
        "type": "object",
        "properties": {
            "site_name": {
                "type": "string",
                "description": "O endereço que o usuário quer consultar.",
            },
        },
        "required": ["site_name"],
        "additionalProperties": False
    }
}

In [33]:
tools = [
    {"type": "function", "function": news_function},
    {"type": "function", "function": site_function}
        ]

In [39]:
def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    
    if function_name == "adress_function":
        site_name = arguments.get("site_name")
        print(f"Buscando endereço do site: {site_name}")
        try:
            url = adress_function(site_name)
        except Exception as e:
            print(e)
        response_content = {"site_name": site_name, "url": url}
    
    elif function_name == "create_brochure":
        url = arguments.get("url")
        print(f"Resumindo notícias de: {url}")
        resumo = create_brochure(url)
        response_content = {"url": url, "resumo": resumo}

    else:
        print(f"[ERRO] Função desconhecida chamada: {function_name}")
        response_content = {"error": f"Unknown function {function_name}"}
    
    response = {
        "role": "tool",
        "content": json.dumps(response_content),
        "tool_call_id": tool_call.id
    }
    return response

In [44]:

def chat(message, history):
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # Enquanto o modelo quiser chamar ferramentas
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        response_tool = handle_tool_call(message)
        messages.append(message)
        messages.append(response_tool)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
        print(response)

    return response.choices[0].message.content

In [49]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


Buscando endereço do site: dw
ChatCompletion(id='chatcmpl-CRM5YqCOQE6zwlTqLXbiKalnFebCb', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_x2gBHrmW8QumxmA3M5TEI5nQ', function=Function(arguments='{"url":"https://www.dw.com/pt-br/not%C3%ADcias/s-7111"}', name='create_brochure'), type='function')]))], created=1760635800, model='gpt-4o-mini-2024-07-18', object='chat.completion', service_tier='default', system_fingerprint='fp_560af6e559', usage=CompletionUsage(completion_tokens=33, prompt_tokens=233, total_tokens=266, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
Resumindo notícias de: https://www.dw.com/pt-br/not%C3%